In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight


In [2]:
df=pd.read_csv("test.csv")

In [3]:
df.head()

,Flight_ID,Airline,Departure_City,Arrival_City,Distance,Departure_Time,Arrival_Time,Duration,Aircraft_Type,Number_of_Stops,Day_of_Week,Month_of_Travel,Holiday_Season,Demand,Weather_Conditions,Passenger_Count,Promotion_Type,Fuel_Price
0,F45001,Airline B,Davidstad,Moorebury,3096.0,18:43,0:14,5.52,Boeing 737,1,Saturday,August,Summer,Medium,Clear,110,NaN,0.95
1,F45002,Airline A,Lake Tyler,Camachoberg,8760.0,1:16,13:04,11.80,Airbus A380,1,Thursday,April,NaN,High,Clear,295,Discount,1.05
2,F45003,Airline C,New Carol,West Ryanfurt,6365.0,12:17,21:52,9.59,Boeing 777,1,Sunday,January,NaN,Low,Rain,223,Discount,0.63
3,F45004,Airline A,Richardsonshire,Jordanburgh,7836.0,0:11,10:23,10.21,Airbus A380,0,Thursday,March,NaN,Low,Rain,223,NaN,0.88
4,F45005,Airline B,Tiffanytown,Morganstad,1129.0,3:22,5:13,1.86,Airbus A320,1,Saturday,August,Summer,High,Cloudy,145,Special Offer,1.11


***data cleaning***

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Flight_ID           5000 non-null   object 
 1   Airline             4573 non-null   object 
 2   Departure_City      4961 non-null   object 
 3   Arrival_City        4970 non-null   object 
 4   Distance            4991 non-null   float64
 5   Departure_Time      5000 non-null   object 
 6   Arrival_Time        5000 non-null   object 
 7   Duration            5000 non-null   float64
 8   Aircraft_Type       4992 non-null   object 
 9   Number_of_Stops     5000 non-null   int64  
 10  Day_of_Week         4975 non-null   object 
 11  Month_of_Travel     4966 non-null   object 
 12  Holiday_Season      4013 non-null   object 
 13  Demand              4966 non-null   object 
 14  Weather_Conditions  4951 non-null   object 
 15  Passenger_Count     5000 non-null   int64  
 16  Promot

In [5]:
df.isnull().sum()

Flight_ID                0
Airline                427
Departure_City          39
Arrival_City            30
Distance                 9
Departure_Time           0
Arrival_Time             0
Duration                 0
Aircraft_Type            8
Number_of_Stops          0
Day_of_Week             25
Month_of_Travel         34
Holiday_Season         987
Demand                  34
Weather_Conditions      49
Passenger_Count          0
Promotion_Type        1689
Fuel_Price              10
dtype: int64

In [6]:
df["Airline"].value_counts(dropna=False)

Airline
Airline A    1551
Airline C    1532
Airline B    1490
NaN           427
Name: count, dtype: int64

In [7]:
#Impute missing Airline values with a descriptive baseline category
df['Airline'] = df['Airline'].fillna('Unknown')

In [8]:
df["Holiday_Season"].value_counts(dropna=False)

Holiday_Season
Winter    1017
Spring    1010
Fall       998
Summer     988
NaN        987
Name: count, dtype: int64

In [9]:
# Impute missing promotion values with a clear baseline category
df['Promotion_Type'] = df['Promotion_Type'].fillna('No Promotion')


In [10]:
# Impute missing holiday values with a descriptive baseline category
df['Holiday_Season'] = df['Holiday_Season'].fillna('Regular')


In [11]:
df["Promotion_Type"].value_counts(dropna=False)

Promotion_Type
No Promotion     1689
Special Offer    1666
Discount         1645
Name: count, dtype: int64

In [12]:
#now dropping the remaining miss values
df.dropna(inplace=True)

In [13]:
df.to_csv("airline_cleaned.csv")

In [15]:
df['Aircraft_Type'].value_counts()

Aircraft_Type
Boeing 737     998
Boeing 777     961
Airbus A320    945
Boeing 787     944
Airbus A380    921
Name: count, dtype: int64

In [134]:
df.columns

Index(['Flight_ID', 'Airline', 'Departure_City', 'Arrival_City', 'Distance',
       'Departure_Time', 'Arrival_Time', 'Duration', 'Aircraft_Type',
       'Number_of_Stops', 'Day_of_Week', 'Month_of_Travel', 'Holiday_Season',
       'Demand', 'Weather_Conditions', 'Passenger_Count', 'Promotion_Type',
       'Fuel_Price'],
      dtype='object')

***FEATURE ENGINEERING FOR POWER BI & ML***

In [16]:
#Standard industry scaling capacities

capacity_map = {
    'Airbus A380': 500, 
    'Boeing 777': 300, 
    'Boeing 737': 150, 
    'Airbus A320': 150,
    "Boeing 787":336
}

In [17]:
df['Aircraft_Capacity'] = df['Aircraft_Type'].map(capacity_map)

In [18]:
# Calculate Load Factor % (Passenger Count / Capacity)
df['Load_Factor_Pct'] = round((df['Passenger_Count'] / df['Aircraft_Capacity']) * 100, 2)

#Format Route string for mapping visuals
df['Route'] = df['Departure_City'] + " -> " + df['Arrival_City']

# 4. Extract numeric Departure Hour from HH:MM string format
df['Departure_Hour'] = pd.to_datetime(df['Departure_Time'], format='%H:%M').dt.hour


In [33]:
df['Departure_Hour'].head()

0    18
1     1
2    12
3     0
4     3
Name: Departure_Hour, dtype: int32

***handlling high-cardinality categorical variables***

In [19]:
# Calculate the average passenger count for each route
route_target_mean = df.groupby('Route')['Passenger_Count'].mean()

# Map those averages back to the dataset
df['Route_Encoded'] = df['Route'].map(route_target_mean)

# Drop the old text column; 'Route_Encoded' is now a single, powerful numeric feature!


To prevent high-cardinality feature explosion from 4,769 unique routes, I implemented Target Encoding to capture route-specific demand metrics within a single numeric dimension

***using Sine and Cosine transformations so the model understands seasonal proximity***

In [20]:
# 1. Map text months to standard numbers 1-12
month_map = {
    'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
    'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12
}
df['Month_Num'] = df['Month_of_Travel'].map(month_map)

# 2. Apply Cyclical Sine/Cosine transformation
df['Month_Sin'] = np.sin(2 * np.pi * df['Month_Num'] / 12.0)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month_Num'] / 12.0)

# Now drop 'Month_of_Travel' and 'Month_Num'. Use 'Month_Sin' and 'Month_Cos' in your model!


In [21]:
# one hot encoding dummy variables
df = pd.get_dummies(df, columns=['Holiday_Season', 'Weather_Conditions',"Aircraft_Type","Promotion_Type","Airline"], drop_first=True, dtype=int)


***Cyclical Day Transformation***

In [22]:

# 1. Map text days to standard sequential numbers 1-7
day_map = {
    'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 
    'Thursday': 4, 'Friday': 5, 'Saturday': 6, 'Sunday': 7
}
df['Day_Num'] = df['Day_of_Week'].map(day_map)

# 2. Apply Cyclical Sine/Cosine transformation (7 days in a week loop)
df['Day_Sin'] = np.sin(2 * np.pi * df['Day_Num'] / 7.0)
df['Day_Cos'] = np.cos(2 * np.pi * df['Day_Num'] / 7.0)

# 3. Drop the old columns before training the model




In [23]:
# Demand_Code target column 
demand_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['Demand_Code'] = df['Demand'].map(demand_map)


***MODEL BUILDING***

In [24]:

# This completely stops your 4,769 routes from leaking data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['Demand_Code'])

# Calculate the baseline metrics using ONLY the training data slice
global_mean = train_df['Passenger_Count'].mean()
route_stats = train_df.groupby('Route')['Passenger_Count'].agg(['count', 'mean'])

# smoothing weight (m=10 forces weak routes to trust the global mean)
m = 10 
smoothed_vals = ((route_stats['count'] * route_stats['mean']) + (m * global_mean)) / (route_stats['count'] + m)
route_encoded_dict = smoothed_vals.to_dict()

# 4. Map the clean numeric values back to BOTH training and testing slices
train_df['Route_Encoded'] = train_df['Route'].map(route_encoded_dict).fillna(global_mean)
test_df['Route_Encoded'] = test_df['Route'].map(route_encoded_dict).fillna(global_mean)

print("Smoothed Route Encoding successfully plugged in!")

# List all columns that  must be  dropped from X so the model doesn't cheat or crash
drop_cols = [
    'Flight_ID', 'Departure_City', 'Arrival_City', 'Route', 
    'Departure_Time', 'Arrival_Time', 'Day_of_Week', 'Month_of_Travel', 
    'Month_Num', 'Day_Num', 'Demand', 'Demand_Code', 
    'Passenger_Count', 'Load_Factor_Pct'
]

#  X and y for training
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['Demand_Code']

# X and y for testing
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['Demand_Code']



Smoothed Route Encoding successfully plugged in!


***BASELINE MODEL (Simple Decision Tree)***

In [25]:
baseline_model = DecisionTreeClassifier(max_depth=5, random_state=42)
baseline_model.fit(X_train, y_train)

# Generate Baseline predictions
y_pred_baseline = baseline_model.predict(X_test)


***OPTIMIZED MODEL (Balanced Random Forest)***

In [26]:
# Ensembles multiple trees and balances weights to accurately predict minority classes
optimized_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42
)
optimized_model.fit(X_train, y_train)

# Generate Optimized predictions
y_pred_optimized = optimized_model.predict(X_test)

***THE OPTIMIZED XGBOOST MODEL***

In [27]:
#Calculate sample weights to balance your 66% 'Low' demand data skew
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Initialize XGBoost for Multi-class classification (Low=0, Medium=1, High=2)
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    objective='multi:softprob',  # Outputs class probabilities
    random_state=42
)

# Train XGBoost passing our balancing weights into the fit method
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test)

***THE PERFORMANCE COMPARISON MATRIX***

In [28]:
# Calculate weighted metrics across all 3 demand tiers (Low, Medium, High)
metrics_summary = {
    'Metric': ['Overall Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1-Score (Weighted)'],
    'Baseline Model': [
        f"{accuracy_score(y_test, y_pred_baseline):.2%}",
        f"{precision_score(y_test, y_pred_baseline, average='weighted'):.2%}",
        f"{recall_score(y_test, y_pred_baseline, average='weighted'):.2%}",
        f"{f1_score(y_test, y_pred_baseline, average='weighted'):.2%}"
    ],
    'Optimized Random Forest': [
        f"{accuracy_score(y_test, y_pred_optimized):.2%}",
        f"{precision_score(y_test, y_pred_optimized, average='weighted'):.2%}",
        f"{recall_score(y_test, y_pred_optimized, average='weighted'):.2%}",
        f"{f1_score(y_test, y_pred_optimized, average='weighted'):.2%}"
    ],
'Optimized XGBoost': [
        f"{accuracy_score(y_test, y_pred_xgb):.2%}",
        f"{precision_score(y_test, y_pred_xgb, average='weighted'):.2%}",
        f"{recall_score(y_test, y_pred_xgb, average='weighted'):.2%}",
        f"{f1_score(y_test, y_pred_xgb, average='weighted'):.2%}"
    ]

}

comparison_df = pd.DataFrame(metrics_summary)

In [29]:
print("\n" + "="*50)
print(" MODEL COMPARISON MATRIX")
print("="*50)
print(comparison_df.to_string(index=False))
print("="*50 + "\n")

# Print detailed breakdowns to isolate how well they predicted "High" and "Medium" flights
print("DETAILED BASELINE MODEL REPORT:")
print(classification_report(y_test, y_pred_baseline, target_names=['Low', 'Medium', 'High']))

print("\n DETAILED OPTIMIZED Random Forest:")
print(classification_report(y_test, y_pred_optimized, target_names=['Low', 'Medium', 'High']))

print("\n DETAILED OPTIMIZED XGBoost:")
print(classification_report(y_test, y_pred_xgb, target_names=['Low', 'Medium', 'High']))





 MODEL COMPARISON MATRIX
              Metric Baseline Model Optimized Random Forest Optimized XGBoost
    Overall Accuracy         70.44%                  70.44%            70.13%
Precision (Weighted)         67.55%                  79.49%            77.29%
   Recall (Weighted)         70.44%                  70.44%            70.13%
 F1-Score (Weighted)         65.33%                  73.27%            72.75%

DETAILED BASELINE MODEL REPORT:
              precision    recall  f1-score   support

         Low       0.78      0.91      0.84       629
      Medium       0.46      0.52      0.49       193
        High       0.50      0.01      0.01       132

    accuracy                           0.70       954
   macro avg       0.58      0.48      0.45       954
weighted avg       0.68      0.70      0.65       954


 DETAILED OPTIMIZED Random Forest:
              precision    recall  f1-score   support

         Low       0.99      0.79      0.88       629
      Medium       0.46  

In [96]:
comparison_df

,Metric,Baseline Model,Optimized Random Forest,Optimized XGBoost
0,Overall Accuracy,70.44%,69.81%,70.75%
1,Precision (Weighted),67.55%,78.91%,77.77%
2,Recall (Weighted),70.44%,69.81%,70.75%
3,F1-Score (Weighted),65.33%,72.71%,73.29%


***Saving data for reporting***

In [98]:
# Generate predictions across both splits using XGBoost
train_df['Predicted_Demand_Code'] = xgb_model.predict(X_train)
test_df['Predicted_Demand_Code'] = xgb_model.predict(X_test)

# Combine datasets seamlessly
df_final = pd.concat([train_df, test_df], axis=0)

# Decode target and prediction codes back into strings for business visuals
reverse_demand_map = {0: 'Low', 1: 'Medium', 2: 'High'}
df_final['Predicted_Demand'] = df_final['Predicted_Demand_Code'].map(reverse_demand_map)
df_final['Model_Correct'] = df_final['Demand'] == df_final['Predicted_Demand']

# Save down the master asset for Power BI Desktop
df_final.to_csv('flight_demand_bi_ready.csv', index=False)
print("Success! Master dashboard asset exported as 'flight_demand_bi_ready.csv'")


Success! Master dashboard asset exported as 'flight_demand_bi_ready.csv'


In [118]:
test_data.dropna(inplace=True)

In [135]:
df_final.columns

Index(['Flight_ID', 'Departure_City', 'Arrival_City', 'Distance',
       'Departure_Time', 'Arrival_Time', 'Duration', 'Number_of_Stops',
       'Day_of_Week', 'Month_of_Travel', 'Demand', 'Passenger_Count',
       'Fuel_Price', 'Aircraft_Capacity', 'Load_Factor_Pct', 'Route',
       'Departure_Hour', 'Route_Encoded', 'Month_Num', 'Month_Sin',
       'Month_Cos', 'Day_Num', 'Day_Sin', 'Day_Cos', 'Holiday_Season_Regular',
       'Holiday_Season_Spring', 'Holiday_Season_Summer',
       'Holiday_Season_Winter', 'Weather_Conditions_Cloudy',
       'Weather_Conditions_Rain', 'Weather_Conditions_Snow',
       'Aircraft_Type_Airbus A380', 'Aircraft_Type_Boeing 737',
       'Aircraft_Type_Boeing 777', 'Aircraft_Type_Boeing 787',
       'Promotion_Type_No Promotion', 'Promotion_Type_Special Offer',
       'Airline_Airline B', 'Airline_Airline C', 'Airline_Unknown',
       'Demand_Code', 'Predicted_Demand_Code', 'Predicted_Demand',
       'Model_Correct'],
      dtype='object')

In [137]:
cc=pd.read_csv("flight_demand_bi_ready.csv")

In [138]:
cc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4769 entries, 0 to 4768
Data columns (total 44 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Flight_ID                     4769 non-null   object 
 1   Departure_City                4769 non-null   object 
 2   Arrival_City                  4769 non-null   object 
 3   Distance                      4769 non-null   float64
 4   Departure_Time                4769 non-null   object 
 5   Arrival_Time                  4769 non-null   object 
 6   Duration                      4769 non-null   float64
 7   Number_of_Stops               4769 non-null   int64  
 8   Day_of_Week                   4769 non-null   object 
 9   Month_of_Travel               4769 non-null   object 
 10  Demand                        4769 non-null   object 
 11  Passenger_Count               4769 non-null   int64  
 12  Fuel_Price                    4769 non-null   float64
 13  Air

In [140]:
cc[["Model_Correct","Predicted_Demand"]].head(10)

,Model_Correct,Predicted_Demand
0,True,Low
1,True,Medium
2,False,High
3,True,Medium
4,True,Low
5,True,Low
6,False,Medium
7,True,Low
8,True,Low
9,True,Low


In [142]:
cc["Model_Correct"].sum()/len(cc)

np.float64(0.8519605787376808)